# Structured output

# 结构化输出 有四种方式  Pydantic[推荐]，TypedDict，JSON Schema , Data classess

## Pydantic

In [1]:
import os
from dotenv import load_dotenv
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain.agents import create_agent

load_dotenv()

llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    api_key=os.getenv("GOOGLE_API_KEY"),
)


In [2]:
from pydantic import BaseModel,Field

# 定义结构化数据 
class Movie(BaseModel):
    title:str = Field(description="The title of the movie") # 电影名
    year:int = Field(description="This year the movie was released") # 发布时间
    director:str = Field(description="The director of the movie") # 电影导演
    rating:float = Field(description="The movies rating out of 10")  # 评分

In [3]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import PydanticOutputParser

parser = PydanticOutputParser(pydantic_object=Movie)

prompt = ChatPromptTemplate.from_messages([
    ("system", "你是电影推荐助手。只输出符合格式的 JSON,不要任何额外文字、不要寒暄。\n{format_instructions}"),
    ("user", "{query}"),
]).partial(format_instructions=parser.get_format_instructions())

chain = prompt | llm | parser
print(chain.invoke({"query": "给我推荐一部电影"}))

title='肖申克的救赎' year=1994 director='弗兰克·德拉邦特' rating=9.3


# Message Output AlongSide parsed structure

In [4]:
from pydantic import BaseModel, Field
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import PydanticOutputParser

class Movie(BaseModel):
    title: str = Field(..., description="The title of the movie")
    year: int = Field(..., description="The year the movie was released")
    director: str = Field(..., description="The director of the movie")
    rating: float = Field(..., description="The movie's rating out of 10")

parser = PydanticOutputParser(pydantic_object=Movie)

prompt = ChatPromptTemplate.from_messages([
    ("system",
     "你是电影信息助手。只输出符合 schema 的 JSON,绝对不要寒暄、不要解释、不要任何多余文字。\n{format_instructions}"),
    ("user", "{query}"),
]).partial(format_instructions=parser.get_format_instructions())

# 关键:链路只到 llm,先拿原始 AIMessage,不接 parser
raw_chain = prompt | llm

ai_message = raw_chain.invoke({"query": "给我盗梦空间的信息"})
print(ai_message)
raw_text = ai_message.content

print("=== RAW 原始输出 ===")
print(raw_text)

print("\n=== 尝试解析 ===")
try:
    movie = parser.parse(raw_text)
    print("解析成功:", movie)
except Exception as e:
    print("解析失败:", e)

content='```json\n{"title": "Inception", "year": 2010, "director": "Christopher Nolan", "rating": 8.8}\n```' additional_kwargs={} response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash', 'safety_ratings': [], 'model_provider': 'google_genai'} id='lc_run--019f162f-ef20-7c13-8514-c012c7cb0131-0' tool_calls=[] invalid_tool_calls=[] usage_metadata={'input_tokens': 281, 'output_tokens': 157, 'total_tokens': 438, 'input_token_details': {'cache_read': 0}, 'output_token_details': {'reasoning': 121}}
=== RAW 原始输出 ===
```json
{"title": "Inception", "year": 2010, "director": "Christopher Nolan", "rating": 8.8}
```

=== 尝试解析 ===
解析成功: title='Inception' year=2010 director='Christopher Nolan' rating=8.8


In [ ]:
from pydantic import BaseModel, Field

class Actor(BaseModel):
    name: str
    role: str 

class MovieDetails(BaseModel):
    tile: str
    year: int 
    cast: list[Actor]
    genres: list[str]
    budget: float | None = Field(None, description="Budget in millions USD")

modle_with_structure = llm.with_structured_output(MovieDetails)
resp = modle_with_structure.invoke("Provide details about the movie Inception")
resp

MovieDetails(tile='Inception', year=2010, cast=[Actor(name='Leonardo DiCaprio', role='Dom Cobb'), Actor(name='Joseph Gordon-Levitt', role='Arthur'), Actor(name='Elliot Page', role='Ariadne'), Actor(name='Tom Hardy', role='Eames'), Actor(name='Ken Watanabe', role='Saito'), Actor(name='Cillian Murphy', role='Robert Fischer'), Actor(name='Marion Cotillard', role='Mal')], genres=['Science Fiction', 'Action', 'Thriller', 'Mystery'], budget=160.0)

## TypedDict

In [8]:
from typing_extensions import TypedDict, Annotated

class MovieDict(TypedDict):
    """Movie Details"""
    titile: Annotated[str, ..., "The title of movie"]
    year: Annotated[int, ..., "The year the movie was released"]
    director: Annotated[str, ..., "The director of the movie"]
    rating: Annotated[float, ..., "The movie's rating out of 10"]

model_with_dict = llm.with_structured_output(MovieDict)


resp = model_with_dict.invoke("请给我提供一个电影详细信息")
resp

{'titile': 'Inception',
 'year': 2010,
 'director': 'Christopher Nolan',
 'rating': 8.8}

In [11]:
# 嵌套
from typing_extensions import TypedDict, Annotated

class Actor(TypedDict):
    name: str
    role: str 

class MovieDict(TypedDict):
    titile: Annotated[str, ..., "The title of movie"]
    year: Annotated[int, ..., "The year the movie was released"]
    cast: list[Actor]
    genres: list[str]
    budget: Annotated[float, ..., "Budget in millions USD"]


model_with_dict = llm.with_structured_output(MovieDict)


resp = model_with_dict.invoke("请给我提供一个电影详细信息")
resp

{'titile': 'The Quantum Leap',
 'year': 2023,
 'cast': [{'name': 'Alice Johnson', 'role': 'Dr. Eleanor Vance'},
  {'name': 'Bob Williams', 'role': 'Commander Alex Stone'},
  {'name': 'Charlie Davis', 'role': 'Dr. Kenji Tanaka'}],
 'genres': ['Science Fiction', 'Action', 'Thriller'],
 'budget': 250.5}

# DataClasses

In [17]:
from pydantic import BaseModel, Field
from langchain.agents import create_agent

class ContactInfo(BaseModel):
    """Contact information for a person"""
    name: str = Field(description="The name of the person")
    email: str = Field(description="The email of the person")
    phone: str = Field(description="The phone number of the person")

# 创建 agent 
agent = create_agent(
    model=llm,
    response_format=ContactInfo,
)

resp = agent.invoke(
    {"messages": [{"role": "user", "content": "提供联系人信息 从 John Doe, 123@gmail.com, +1(246)5567-7424"}]}
)

resp

{'messages': [HumanMessage(content='提供联系人信息 从 John Doe, 123@gmail.com, +1(246)5567-7424', additional_kwargs={}, response_metadata={}, id='af682ab9-7ee6-47f7-9357-ffef8f1bc4d7'),
  AIMessage(content='{"name":"John Doe","email":"123@gmail.com","phone":"+1(246)5567-7424"}', additional_kwargs={}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019f165e-c547-7d63-8a18-7a5c590fff2a-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 34, 'output_tokens': 175, 'total_tokens': 209, 'input_token_details': {'cache_read': 0}, 'output_token_details': {'reasoning': 140}})],
 'structured_response': ContactInfo(name='John Doe', email='123@gmail.com', phone='+1(246)5567-7424')}